# Project Resonova — Phase 5 Evaluation & Benchmarking Notebook
This notebook runs the public benchmarks for Project Resonova (ASR, Translation, Voice Cloning, Prosody Style Preservation) using RAVDESS and FLORES-200 datasets.

Run each cell sequentially. All outputs are written to Google Drive.

## Cell 1: Environment Setup
Mount Google Drive, pull repository modifications, and install required dependencies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Navigate to drive workspace folder
%cd /content/drive/MyDrive/

import os
if not os.path.exists('resonova'):
    !git clone https://github.com/SAK-SHI14/resonova.git

%cd resonova
!git pull

# Install project in editable mode
!pip install -e .

# Install specific benchmarking packages
!pip install resemblyzer sacrebleu transformers nltk soundfile pydub librosa

## Cell 2: Download & Extract Benchmark Datasets
This cell downloads the RAVDESS Speech subset (~215MB) and the FLORES-200 English/Hindi evaluation sentence datasets directly into Google Drive.

In [ ]:
import urllib.request
import zipfile
import os

eval_data_dir = "/content/drive/MyDrive/resonova_checkpoints/eval_data"
os.makedirs(eval_data_dir, exist_ok=True)

# 1. RAVDESS Speech Subset Download
ravdess_url = "https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip?download=1"
ravdess_zip = os.path.join(eval_data_dir, "Audio_Speech_Actors_01-24.zip")
ravdess_dir = os.path.join(eval_data_dir, "ravdess")

if not os.path.exists(ravdess_zip):
    print("Downloading RAVDESS dataset (~215MB)... This may take 1-3 minutes.")
    urllib.request.urlretrieve(ravdess_url, ravdess_zip)
    print("RAVDESS ZIP downloaded successfully.")

if not os.path.exists(ravdess_dir):
    print("Extracting RAVDESS dataset...")
    with zipfile.ZipFile(ravdess_zip, 'r') as zf:
        zf.extractall(ravdess_dir)
    print("RAVDESS extraction complete.")

# 2. FLORES-200 English-Hindi devtest sets
flores_en_url = "https://raw.githubusercontent.com/facebookresearch/flores/main/flores200/devtest/eng_Latn.devtest"
flores_hi_url = "https://raw.githubusercontent.com/facebookresearch/flores/main/flores200/devtest/hin_Deva.devtest"
flores_en_path = os.path.join(eval_data_dir, "eng_Latn.devtest")
flores_hi_path = os.path.join(eval_data_dir, "hin_Deva.devtest")

if not os.path.exists(flores_en_path):
    print("Downloading FLORES-200 English sentences...")
    urllib.request.urlretrieve(flores_en_url, flores_en_path)

if not os.path.exists(flores_hi_path):
    print("Downloading FLORES-200 Hindi sentences...")
    urllib.request.urlretrieve(flores_hi_url, flores_hi_path)

print("All datasets verified and ready!")

## Cell 3: Run RAVDESS Emotion Preservation Evaluation
Processes a stratified subset of 20 RAVDESS clips across targets. Measures Speech Emotion Recognition (SER) detector baseline accuracy and the pipeline's overall preservation rate on dubbed outputs.

In [ ]:
from resonova.eval.benchmark import run_ravdess_emotion_eval

# Configure environment paths for local/subprocesses if needed
# (Note: Wav2Lip env variables are only needed for full video dubbing runs, audio-only uses standard modules)

ravdess_results = run_ravdess_emotion_eval(ravdess_dir, n_samples=20)
print("=== RAVDESS EMOTION BENCHMARK RESULTS ===")
print(f"- SER Classifier Baseline Accuracy : {ravdess_results['ser_accuracy']:.2%}")
print(f"- Emotion Preservation Success Rate : {ravdess_results['emotion_preservation_rate']:.2%}")
print(f"- Samples evaluated                 : {ravdess_results['samples_tested']}")

## Cell 4: Run FLORES English-Hindi Translation Evaluation
Translates 100 devtest sentences using the IndicTrans2 model wrapper, comparing output BLEU/chrF scores against published research benchmarks.

In [ ]:
from resonova.eval.benchmark import run_flores_translation_eval

flores_results = run_flores_translation_eval(
    flores_en_path=flores_en_path,
    flores_hi_path=flores_hi_path,
    n_samples=100
)

print("=== FLORES-200 TRANSLATION BENCHMARK RESULTS ===")
print(f"- Our Average Translation BLEU Score : {flores_results['bleu']:.4f}")
print(f"- Our Average Translation chrF Score : {flores_results['chrf']:.4f}")
print(f"- Published IndicTrans2 Baseline BLEU: {flores_results['published_baseline_bleu'] / 100.0:.4f}")
print(f"- Difference over Published Baseline : {flores_results['our_vs_baseline']:+.4f}")

## Cell 5: Run Speaker Similarity Evaluation
Runs the voice cloning pipeline over target source clips, using Resemblyzer to verify speaker identity preservation.

In [ ]:
from resonova.eval.benchmark import run_speaker_similarity_eval

similarity_results = run_speaker_similarity_eval("samples")

print("=== SPEAKER SIMILARITY RESULTS ===")
print(f"- Mean Speaker Embedding Cosine Similarity : {similarity_results['mean_similarity']:.4f}")
print(f"- Max Similarity                           : {similarity_results['max_similarity']:.4f}")
print(f"- Min Similarity                           : {similarity_results['min_similarity']:.4f}")
print(f"- Total Clips Checked                      : {similarity_results['clips_tested']}")

## Cell 6: Run Ablation Study (ON vs. OFF)
Runs the pipeline with style conditioning ON and OFF (using a neutral fallback profile) to prove the effectiveness of the prosody preservation layers.

In [ ]:
from resonova.eval.benchmark import run_ablation_study

ablation_results = run_ablation_study("samples")

print("=== ABLATION STUDY COMPARISON ===")
print(f"- Clips tested: {ablation_results['clips_tested']}")
print("| Evaluated Metric | Style ON | Style OFF | Improvement (Delta) |")
print("|------------------|----------|-----------|---------------------|")
print(f"| Speaker Sim      | {ablation_results['resonova_conditioned']['speaker_similarity']:.4f} | {ablation_results['baseline']['speaker_similarity']:.4f} | {ablation_results['improvement']['speaker_similarity']:+.4f} |")
print(f"| Pitch/Energy (r) | {ablation_results['resonova_conditioned']['emotion_agreement']:.4f} | {ablation_results['baseline']['emotion_agreement']:.4f} | {ablation_results['improvement']['emotion_agreement']:+.4f} |")
print(f"| SER Agreement    | {ablation_results['resonova_conditioned']['ser_agreement']:.2%} | {ablation_results['baseline']['ser_agreement']:.2%} | {ablation_results['improvement']['ser_agreement']:+.2%} |")

## Cell 7: Generate Final Evaluation Report
Combines all automated metrics with human-evaluation survey results to output a complete project validation report.

In [ ]:
from resonova.eval.benchmark import generate_eval_report

# Put here your collected human-evaluation scores (10 evaluators average)
human_results = {
    "n_evaluators": 10,
    "voice_match_rate": 0.86, # e.g. 86%
    "emotion_realism_rate": 0.81, # e.g. 81%
    "mean_quality_score": 4.2 # e.g. 4.2 / 5.0
}

report_path = "docs/eval_report.md"
generate_eval_report(
    ravdess_results=ravdess_results,
    flores_results=flores_results,
    similarity_results=similarity_results,
    ablation_results=ablation_results,
    human_eval_results=human_results,
    output_path=report_path
)

# Copy output markdown to Drive for persistence
!cp docs/eval_report.md /content/drive/MyDrive/resonova_checkpoints/eval_report.md

print("Evaluation Report generated successfully and backed up to Drive!")